# Kalman filter test bench: Thymio + phone camera

Estimates the pose $(x, y, \theta)$ of the Thymio with an Extended Kalman Filter (`kalman.py`):
- **prediction** with the odometry of the measured wheel speeds, at every loop;
- **update** with the pose measured by the phone camera (`vision.py`), whenever the robot is visible.

Setup: `README.md`. Theory and choice of the constants: `THEORIE_KALMAN.md`. Run the cells in order.

## 1. Settings

In [1]:
import time

import cv2
import matplotlib.pyplot as plt
import numpy as np
from tdmclient import ClientAsync, aw

import vision
from kalman import ExtendedKalmanFilter

CAMERA_SOURCE = "http://172.20.10.3:4747/video"  # DroidCam app

WHITE_THRESHOLD = 90  # gray level (0-255) above which a pixel counts as the white robot, tuned in section 3

# Thymio (course Thymio cheat sheet)
THYMIO_SIZE_MM = 111    # mean side of the robot seen from above (110 x 112 mm), gives the mm/px scale
WHEEL_DISTANCE_MM = 95  # b, distance between the two wheels

# Defaults, replaced by the measurements of section 4
SPEED_TO_MM_S = 0.43             # Thymio speed unit -> mm/s (Exercise 8; cheat sheet: 500 = ~20 cm/s)
SPEED_VARIANCE = 5.4             # var(v) [mm^2/s^2] (Exercise 8 data, converted to mm/s)
ANGULAR_SPEED_VARIANCE = 1.6e-3  # var(omega) [rad^2/s^2] (Exercise 8 data)
CAMERA_VARIANCES = [1.0, 1.0, np.radians(1.0) ** 2]  # var(x) [mm^2], var(y) [mm^2], var(theta) [rad^2]

TEST_MOTOR_TARGETS = (100, 10)  # left, right [Thymio units]: circle of ~40 cm diameter
GREEN, RED, BLUE = (0, 200, 0), (0, 0, 255), (255, 80, 0)  # BGR colours: camera, odometry only, Kalman

## 2. Connect to the Thymio

Start the Thymio Device Manager and connect the robot (USB cable or RF dongle) first.
This cell loads on the robot a tiny Aseba program that applies the motor targets sent by the PC
(course notebook *Control your Thymio in Python*, section 6.4), then watches the measured wheel speeds.

In [2]:
THYMIO_PROGRAM = """
onevent speed
    motor.left.target = event.args[0]
    motor.right.target = event.args[1]
"""

client = ClientAsync()
node = aw(client.wait_for_node())
aw(node.lock())
aw(node.register_events([("speed", 2)]))
aw(node.compile(THYMIO_PROGRAM))
aw(node.run())
aw(node.wait_for_variables({"motor.left.speed", "motor.right.speed"}))


def set_motor_targets(left, right):
    """Motor targets in Thymio units (-500 to 500)."""
    aw(node.send_events({"speed": [left, right]}))


def read_wheel_speeds():
    """Latest measured wheel speeds (left, right) in Thymio units."""
    client.process_waiting_messages()  # receive the variables sent by the robot since the last call
    return node.v.motor.left.speed, node.v.motor.right.speed


set_motor_targets(0, 0)
print("Connected to", node)

Connected to Node 6ea309b4-a7a3-4cff-8563-7694387de426


## 3. Camera check, white threshold and pixel → mm scale

Two windows open: the camera image with the detected pose (green arrow, at the wheel axle) and the **mask**:
the pixels brighter than the white threshold. Move the slider of the mask window until **only the robot is white**:
too low, the background turns white; too high, the robot gets holes and is lost. Choose the middle of the range that works.
Move the robot by hand to check the arrow, then leave it still, click on a window and press **q**:
the robot size is then measured on 30 frames to get the pixel → mm scale.

In [3]:
camera = vision.open_camera(CAMERA_SOURCE)

RuntimeError: Cannot open the camera 'http://172.20.10.3:4747/video'

Exception in thread Thread-3:
Traceback (most recent call last):
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.13_3.13.3824.0_x64__qbz5n2kfra8p0\Lib\threading.py", line 1044, in _bootstrap_inner
    self.run()
    ~~~~~~~~^^
  File "C:\Users\timvi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tdmclient\tcp.py", line 73, in run
    packet = self.read_packet()
  File "C:\Users\timvi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tdmclient\tcp.py", line 66, in read_packet
    raise error
  File "C:\Users\timvi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tdmclient\tcp.py", line 59, in read_packet
    packet_len = self.read_uint32()
  File "C:\Users\timvi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCa

In [ ]:
cv2.namedWindow("mask")
cv2.createTrackbar("white threshold", "mask", WHITE_THRESHOLD, 255, lambda value: None)
while True:
    WHITE_THRESHOLD = cv2.getTrackbarPos("white threshold", "mask")
    frame = vision.read_frame(camera)
    detection = vision.detect_robot(frame, WHITE_THRESHOLD)
    if detection is not None:
        vision.draw_pose(frame, detection.x, detection.y, detection.angle, GREEN)
        text = f"x {detection.x:.0f} px  y {detection.y:.0f} px  angle {np.degrees(detection.angle):.0f} deg"
        cv2.putText(frame, text, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, GREEN, 2)
    cv2.imshow("camera", frame)
    cv2.imshow("mask", vision.segment_bright_regions(frame, WHITE_THRESHOLD))
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break
cv2.destroyAllWindows()
print(f"WHITE_THRESHOLD = {WHITE_THRESHOLD}  (paste it in section 1 to keep it)")

# Pixel -> mm scale from the known robot size, measured on 30 frames of the still robot
scale_detections = [vision.detect_robot(vision.read_frame(camera), WHITE_THRESHOLD) for _ in range(30)]
robot_sizes = [d.size for d in scale_detections if d is not None]
assert robot_sizes, "The robot was not detected: tune the threshold again, see README > Troubleshooting"
MM_PER_PIXEL = THYMIO_SIZE_MM / np.median(robot_sizes)
print(f"Robot size: {np.median(robot_sizes):.0f} px -> MM_PER_PIXEL = {MM_PER_PIXEL:.3f}")


def pose_in_mm(detection):
    """Camera detection [px, px, rad] -> pose [mm, mm, rad]."""
    return np.array([detection.x * MM_PER_PIXEL, detection.y * MM_PER_PIXEL, detection.angle])

## 4. Measure the constants of the filter

As in Exercise 8, the variances are measured on recorded data
(course *Project Common Pitfalls*: never keep default values):
1. robot still for 3 s: camera noise $R$;
2. robot driving straight for 4 s: speed conversion factor and odometry noise $Q$.

Place the robot in the middle of the image with about 25 cm of free space in front of it.

In [ ]:
def record(duration, left_target, right_target):
    """Apply constant motor targets during `duration` s; return times, camera detections and wheel speeds."""
    set_motor_targets(left_target, right_target)
    times, detections, left_speeds, right_speeds = [], [], [], []
    start = time.monotonic()
    while time.monotonic() - start < duration:
        detections.append(vision.detect_robot(vision.read_frame(camera), WHITE_THRESHOLD))
        left_speed, right_speed = read_wheel_speeds()
        times.append(time.monotonic())
        left_speeds.append(left_speed)
        right_speeds.append(right_speed)
    set_motor_targets(0, 0)
    return np.array(times), detections, np.array(left_speeds, float), np.array(right_speeds, float)


_, still_detections, _, _ = record(duration=3.0, left_target=0, right_target=0)
times, detections, left_speeds, right_speeds = record(duration=4.0, left_target=100, right_target=100)

In [ ]:
# R: variance of the camera pose of the still robot
still_poses = np.array([pose_in_mm(d) for d in still_detections if d is not None])
assert len(still_poses) > 10, "The still robot was not seen by the camera"
still_poses[:, 2] = np.unwrap(still_poses[:, 2])  # no jump between -pi and pi
CAMERA_VARIANCES = np.var(still_poses, axis=0)

# Speed conversion: distance seen by the camera / integral of the measured speed [unit * s]
seen = [k for k, d in enumerate(detections) if d is not None]
assert len(seen) > 10, "The moving robot was not seen by the camera"
first, last = seen[0], seen[-1]
start_xy, end_xy = pose_in_mm(detections[first])[:2], pose_in_mm(detections[last])[:2]
camera_distance = np.hypot(*(end_xy - start_xy))
mean_speeds = (left_speeds + right_speeds) / 2
SPEED_TO_MM_S = camera_distance / np.sum(mean_speeds[first + 1:last + 1] * np.diff(times[first:last + 1]))

# Q: variance of the measured speeds once the robot cruises (first second: acceleration)
cruising = times > times[0] + 1.0
SPEED_VARIANCE = np.var(mean_speeds[cruising] * SPEED_TO_MM_S)
ANGULAR_SPEED_VARIANCE = np.var((right_speeds - left_speeds)[cruising] * SPEED_TO_MM_S / WHEEL_DISTANCE_MM)

print("Measured constants (paste them in section 1 to keep them):")
print(f"SPEED_TO_MM_S = {SPEED_TO_MM_S:.4f}")
print(f"SPEED_VARIANCE = {SPEED_VARIANCE:.3g}")
print(f"ANGULAR_SPEED_VARIANCE = {ANGULAR_SPEED_VARIANCE:.3g}")
print(f"CAMERA_VARIANCES = [{CAMERA_VARIANCES[0]:.3g}, {CAMERA_VARIANCES[1]:.3g}, {CAMERA_VARIANCES[2]:.3g}]")

plt.figure(figsize=(8, 3))
plt.plot(times - times[0], left_speeds, label="left")
plt.plot(times - times[0], right_speeds, label="right")
plt.axvline(1.0, color="gray", linestyle="--", label="start of the variance window")
plt.xlabel("time [s]")
plt.ylabel("measured speed [Thymio units]")
plt.legend()
plt.show()

## 5. Run the filter

The robot drives a circle. Green: camera, red: odometry only, blue: Kalman estimate with its 95 % ellipse.
- **c**: camera on/off for the filter (hidden camera).
  The green arrow stays visible: it should remain inside the ellipse.
- **q**: stop.

Also try a kidnapping: lift the robot and put it somewhere else.

In [ ]:
def draw_estimate(frame, pose, color, covariance=None):
    """Arrow of a pose [mm, mm, rad] and, if given, its 95 % position ellipse."""
    x_px, y_px = pose[:2] / MM_PER_PIXEL
    vision.draw_pose(frame, x_px, y_px, pose[2], color)
    if covariance is not None:
        vision.draw_covariance_ellipse(frame, x_px, y_px, covariance[:2, :2] / MM_PER_PIXEL**2, color)


def write_lines(frame, lines):
    for row, line in enumerate(lines):
        cv2.putText(frame, line, (10, 25 + 25 * row), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

In [ ]:
detection = None
while detection is None:  # the filter starts from the first camera measurement
    frame = vision.read_frame(camera)
    detection = vision.detect_robot(frame, WHITE_THRESHOLD)
    write_lines(frame, ["Waiting for the robot: put it fully in view, away from the border"])
    cv2.imshow("kalman", frame)
    cv2.waitKey(1)

filter_settings = dict(initial_pose=pose_in_mm(detection), initial_covariance=np.diag(CAMERA_VARIANCES),
                       wheel_distance=WHEEL_DISTANCE_MM, speed_variance=SPEED_VARIANCE,
                       angular_speed_variance=ANGULAR_SPEED_VARIANCE, camera_variances=CAMERA_VARIANCES)
kalman = ExtendedKalmanFilter(**filter_settings)
odometry_only = ExtendedKalmanFilter(**filter_settings)  # never updated: pure odometry, for comparison

history = []
use_camera = True
set_motor_targets(*TEST_MOTOR_TARGETS)
previous_time = time.monotonic()
try:
    while True:
        frame = vision.read_frame(camera)
        left_units, right_units = read_wheel_speeds()
        left_speed, right_speed = left_units * SPEED_TO_MM_S, right_units * SPEED_TO_MM_S
        now = time.monotonic()
        dt = now - previous_time
        previous_time = now

        kalman.predict(left_speed, right_speed, dt)
        odometry_only.predict(left_speed, right_speed, dt)
        detection = vision.detect_robot(frame, WHITE_THRESHOLD)
        camera_pose = None if detection is None else pose_in_mm(detection)
        gain = np.zeros((3, 3))
        if use_camera and camera_pose is not None:
            gain = kalman.update(camera_pose)

        history.append({"time": now, "camera": camera_pose, "camera_used": use_camera,
                        "kalman": kalman.pose.copy(), "covariance": kalman.covariance.copy(),
                        "odometry": odometry_only.pose.copy()})

        if camera_pose is not None:
            draw_estimate(frame, camera_pose, GREEN)
        draw_estimate(frame, odometry_only.pose, RED)
        draw_estimate(frame, kalman.pose, BLUE, kalman.covariance)
        sigma = np.sqrt(np.diag(kalman.covariance))
        x, y, theta = kalman.pose
        write_lines(frame, [
            f"camera {'ON' if use_camera else 'OFF'}   [c] camera on/off   [q] quit",
            f"Kalman x {x:6.0f} mm  y {y:6.0f} mm  theta {np.degrees(theta):5.0f} deg",
            f"sigma  x {sigma[0]:6.2f} mm  y {sigma[1]:6.2f} mm  theta {np.degrees(sigma[2]):5.2f} deg",
            f"gain K x {gain[0, 0]:.2f}  y {gain[1, 1]:.2f}  theta {gain[2, 2]:.2f}",
        ])
        cv2.imshow("kalman", frame)
        key = cv2.waitKey(1) & 0xFF
        if key == ord("c"):
            use_camera = not use_camera
        elif key == ord("q"):
            break
finally:
    set_motor_targets(0, 0)
    cv2.destroyAllWindows()

## 6. Results

In [ ]:
elapsed = np.array([h["time"] for h in history]) - history[0]["time"]
kalman_poses = np.array([h["kalman"] for h in history])
odometry_poses = np.array([h["odometry"] for h in history])
camera_poses = np.array([h["camera"] for h in history if h["camera"] is not None])
sigmas = np.array([np.sqrt(np.diag(h["covariance"])) for h in history])
camera_off = np.array([not h["camera_used"] for h in history])

fig, (ax_path, ax_sigma) = plt.subplots(1, 2, figsize=(14, 5))
ax_path.plot(camera_poses[:, 0], camera_poses[:, 1], ".", color="tab:green", markersize=3, label="camera")
ax_path.plot(odometry_poses[:, 0], odometry_poses[:, 1], color="tab:red", label="odometry only")
ax_path.plot(kalman_poses[:, 0], kalman_poses[:, 1], color="tab:blue", label="Kalman")
ax_path.set(xlabel="x [mm]", ylabel="y [mm]", title="Trajectories")
ax_path.set_aspect("equal")
ax_path.legend()

ax_sigma.plot(elapsed, sigmas[:, 0], label="sigma x [mm]")
ax_sigma.plot(elapsed, sigmas[:, 1], label="sigma y [mm]")
ax_sigma.plot(elapsed, np.degrees(sigmas[:, 2]), label="sigma theta [deg]")
ax_sigma.fill_between(elapsed, 0, 1, where=camera_off, transform=ax_sigma.get_xaxis_transform(),
                      color="gray", alpha=0.3, label="camera off")
ax_sigma.set(xlabel="time [s]", title="Standard deviations of the estimate")
ax_sigma.legend()
plt.show()

**Validation of $Q$** (needs a camera-off period made with the key **c**): when the camera comes back,
the camera position (truth) should lie inside the 95 % ellipse, i.e. $d^2 = e^T \Sigma^{-1} e < 5.99$
with $e$ the position error. If it is outside, $Q$ is too small (systematic odometry errors are not in the
speed variance): multiply `SPEED_VARIANCE` and `ANGULAR_SPEED_VARIANCE` by about $d^2 / 2$ and test again.

In [ ]:
for before, after in zip(history[:-1], history[1:]):
    if not before["camera_used"] and after["camera_used"] and before["camera"] is not None:
        error = before["camera"][:2] - before["kalman"][:2]
        d2 = error @ np.linalg.solve(before["covariance"][:2, :2], error)
        print(f"camera back at t = {after['time'] - history[0]['time']:.1f} s: error {np.hypot(*error):.1f} mm, "
              f"d^2 = {d2:.1f} -> {'inside' if d2 < 5.99 else 'OUTSIDE'} the 95 % ellipse")

## 7. Disconnect

In [ ]:
camera.release()
set_motor_targets(0, 0)
aw(node.unlock())